In [6]:
"""
Stage 2 — Dataset Preprocessing
=================================
Converts bounding box annotations (CCL'20 XML + Mendeley YOLO TXT)
into binary segmentation masks for U-Net training.

Why masks from bounding boxes?
  We don't have pixel-level annotations, so the bounding box region
  is used as a proxy for the infected area. This is called
  "weak supervision" — accepted practice in plant disease imaging.

Output structure:
  stage2_dataset/
    train/
      images/   ← resized original images (512×512 JPG)
      masks/    ← binary masks (512×512 PNG) — white=infected, black=background
    val/
      images/
      masks/
    test/
      images/
      masks/
    dataset_info.json  ← summary stats for your report

Run:
  python stage2_preprocess.py
  python stage2_preprocess.py --dry-run   # preview without writing files
"""

import json
import shutil
import random
import argparse
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np

# ══════════════════════════════════════════════════════════════════════════════
# ❶  CONFIGURE YOUR PATHS
# ══════════════════════════════════════════════════════════════════════════════

CCL20_IMAGES_DIR      = "CCL'20 dataset"        # ← change
CCL20_ANNOTATIONS_DIR = "CCL'20 dataset/train"       # ← change (.xml files)

YOLO_IMG_DIR          = "Lemon Leaf Diseases Dataset and Annotations_Mendeley/Lemon Leaf Diseases Dataset"       # ← change
YOLO_TXT_DIR          = "Lemon Leaf Diseases Dataset and Annotations_Mendeley/Lemon Leaf Diseases - Annotations"       # ← change

OUTPUT_DIR   = "stage2_dataset"
TARGET_SIZE  = 512          # resize all images + masks to this
SPLIT        = (0.70, 0.15, 0.15)
SEED         = 42
IMAGE_EXTS   = {".jpg", ".jpeg", ".png", ".bmp"}

# ══════════════════════════════════════════════════════════════════════════════
# ❷  LABEL MAP  (raw annotation name → unified class)
# ══════════════════════════════════════════════════════════════════════════════

LABEL_MAP = {
    # CCL'20 encoded labels
    "Disease-A":            "Anthracnose",
    "Disease-B":            "Melanose",
    "Disease-C":            "BlackSpot",
    # Common variants
    "Anthracnose":          "Anthracnose",
    "anthracnose":          "Anthracnose",
    "Canker":               "Canker",
    "canker":               "Canker",
    "Citrus_Canker":        "Canker",
    "Citrus Canker":        "Canker",
    "Greening":             "Greening",
    "HLB":                  "Greening",
    "Bacterial Blight":     "BacterialBlight",
    "BacterialBlight":      "BacterialBlight",
    "Curl Virus":           "CurlVirus",
    "CurlVirus":            "CurlVirus",
    "Deficiency Leaf":      "DeficiencyLeaf",
    "DeficiencyLeaf":       "DeficiencyLeaf",
    "Dry Leaf":             "DryLeaf",
    "DryLeaf":              "DryLeaf",
    "Sooty Mould":          "SootyMould",
    "SootyMould":           "SootyMould",
    "Spider Mites":         "SpiderMites",
    "SpiderMites":          "SpiderMites",
    "Healthy Leaf":         "Healthy",
    "Healthy":              "Healthy",
    "Melanose":             "Melanose",
    "BlackSpot":            "BlackSpot",
    "Black_spot":           "BlackSpot",
}

# ══════════════════════════════════════════════════════════════════════════════
# ❸  COLLECTORS
# ══════════════════════════════════════════════════════════════════════════════

def find_image(stem: str, search_dirs: list[Path]) -> Path | None:
    """Search for an image file by stem across multiple directories."""
    for d in search_dirs:
        for ext in IMAGE_EXTS:
            for candidate in d.rglob(stem + ext):
                if candidate.exists():
                    return candidate
    return None


def collect_ccl20(images_dir: str, ann_dir: str) -> list[dict]:
    """
    Parse Pascal VOC XML files from CCL'20.
    Each XML → one record with image path + list of bounding boxes.
    A single image may contain multiple disease lesions.
    """
    img_dir = Path(images_dir)
    ann_dir = Path(ann_dir)

    if not ann_dir.exists():
        print(f"  [MISSING] {ann_dir}")
        return []

    records = []
    skipped_labels = set()
    skipped_images = 0

    for xml_file in sorted(ann_dir.glob("*.xml")):
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()
        except ET.ParseError:
            print(f"  [WARN] Bad XML: {xml_file.name}")
            continue

        # Find matching image
        filename = root.findtext("filename", default="")
        img_path = img_dir / filename if filename else None
        if img_path is None or not img_path.exists():
            img_path = find_image(xml_file.stem, [img_dir])
        if img_path is None:
            skipped_images += 1
            continue

        # Read image dimensions from XML (faster than loading image)
        size_node = root.find("size")
        if size_node is not None:
            w = int(size_node.findtext("width",  "0") or "0")
            h = int(size_node.findtext("height", "0") or "0")
        else:
            w = h = 0

        # If dimensions missing, read from image
        if w == 0 or h == 0:
            img = cv2.imread(str(img_path))
            if img is None:
                skipped_images += 1
                continue
            h, w = img.shape[:2]

        # Parse bounding boxes
        boxes = []
        for obj in root.findall("object"):
            raw   = obj.findtext("name", "").strip()
            label = LABEL_MAP.get(raw)
            if label is None:
                skipped_labels.add(raw)
                continue
            bb = obj.find("bndbox")
            if bb is None:
                continue
            xmin = int(float(bb.findtext("xmin", "0")))
            ymin = int(float(bb.findtext("ymin", "0")))
            xmax = int(float(bb.findtext("xmax", "0")))
            ymax = int(float(bb.findtext("ymax", "0")))
            if xmax > xmin and ymax > ymin:
                boxes.append({
                    "label": label,
                    "xmin": xmin, "ymin": ymin,
                    "xmax": xmax, "ymax": ymax,
                })

        if boxes:
            records.append({
                "image_path": img_path,
                "width": w, "height": h,
                "boxes": boxes,
                "source": "CCL20_XML",
            })

    if skipped_labels:
        for sl in sorted(skipped_labels):
            print(f"  [SKIP label] '{sl}' not in LABEL_MAP")
    if skipped_images:
        print(f"  [SKIP] {skipped_images} images not found on disk")

    print(f"  CCL20 XML  → {len(records)} annotated images  "
          f"({sum(len(r['boxes']) for r in records)} total boxes)")
    return records


def collect_yolo(yolo_img_dir: str, yolo_txt_dir: str) -> list[dict]:
    """
    Parse YOLO TXT annotation files.
    Each line: class_id  cx  cy  bw  bh  (all 0–1 normalised)
    """
    txt_root = Path(yolo_txt_dir)
    img_root = Path(yolo_img_dir)
    if not txt_root.exists():
        print(f"  [MISSING] {txt_root}")
        return []

    classes = ["Anthracnose", "Bacterial Blight", "Citrus Canker", "Curl Virus", "Deficiency Leaf", "Dry Leaf", "Healthy Leaf", "Sooty Mould", "Spider Mites"]
    print(f"  YOLO classes: {classes}")

    records = []
    skipped_labels = set()
    skipped_images = 0

    for txt_file in sorted(txt_root.rglob("*.txt")):
        # Find matching image
        img_path = find_image(txt_file.stem, [img_root])
        if img_path is None:
            skipped_images += 1
            continue

        # Get image dimensions
        img = cv2.imread(str(img_path))
        if img is None:
            skipped_images += 1
            continue
        h, w = img.shape[:2]

        # Parse boxes
        boxes = []
        for line in txt_file.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
            except ValueError:
                continue

            raw   = classes[cls_id] if cls_id < len(classes) else str(cls_id)
            label = LABEL_MAP.get(raw)
            if label is None:
                skipped_labels.add(raw)
                continue

            # Convert normalised → pixel coords
            xmin = max(0, int((cx - bw / 2) * w))
            ymin = max(0, int((cy - bh / 2) * h))
            xmax = min(w, int((cx + bw / 2) * w))
            ymax = min(h, int((cy + bh / 2) * h))

            if xmax > xmin and ymax > ymin:
                boxes.append({
                    "label": label,
                    "xmin": xmin, "ymin": ymin,
                    "xmax": xmax, "ymax": ymax,
                })

        if boxes:
            records.append({
                "image_path": img_path,
                "width": w, "height": h,
                "boxes": boxes,
                "source": "Mendeley_YOLO",
            })

    if skipped_labels:
        for sl in sorted(skipped_labels):
            print(f"  [SKIP label] '{sl}' not in LABEL_MAP")
    if skipped_images:
        print(f"  [SKIP] {skipped_images} images not found on disk")

    print(f"  YOLO TXT   → {len(records)} annotated images  "
          f"({sum(len(r['boxes']) for r in records)} total boxes)")
    return records


# ══════════════════════════════════════════════════════════════════════════════
# ❹  MASK GENERATION
# ══════════════════════════════════════════════════════════════════════════════

def generate_mask(record: dict, target_size: int) -> tuple[np.ndarray, np.ndarray] | tuple[None, None]:
    """
    Loads the image, resizes to target_size × target_size.
    Draws filled white rectangles on a black canvas for each bounding box
    (scaled proportionally to the new size).

    Returns: (resized_image BGR, binary_mask uint8)
      mask pixel values: 0 = background, 255 = infected region
    """
    img = cv2.imread(str(record["image_path"]))
    if img is None:
        return None, None

    orig_h, orig_w = img.shape[:2]

    # Use recorded dimensions if image loaded at different size
    src_w = record["width"]  if record["width"]  > 0 else orig_w
    src_h = record["height"] if record["height"] > 0 else orig_h

    scale_x = target_size / src_w
    scale_y = target_size / src_h

    img_resized = cv2.resize(img, (target_size, target_size),
                             interpolation=cv2.INTER_LINEAR)
    mask = np.zeros((target_size, target_size), dtype=np.uint8)

    for box in record["boxes"]:
        x1 = max(0, int(box["xmin"] * scale_x))
        y1 = max(0, int(box["ymin"] * scale_y))
        x2 = min(target_size - 1, int(box["xmax"] * scale_x))
        y2 = min(target_size - 1, int(box["ymax"] * scale_y))
        if x2 > x1 and y2 > y1:
            mask[y1:y2, x1:x2] = 255

    return img_resized, mask


def compute_severity(mask: np.ndarray) -> dict:
    """
    Computes infected area percentage from mask and assigns severity label.
    Used for dataset_info.json — gives severity distribution stats.
    """
    infected_pct = (mask > 127).mean() * 100
    if infected_pct < 5:
        severity = "Healthy"
    elif infected_pct < 25:
        severity = "Mild"
    elif infected_pct < 50:
        severity = "Moderate"
    else:
        severity = "Severe"
    return {"infected_pct": round(float(infected_pct), 2), "severity": severity}


# ══════════════════════════════════════════════════════════════════════════════
# ❺  SPLIT + WRITE
# ══════════════════════════════════════════════════════════════════════════════

def write_split(records: list[dict], split_name: str,
                out_root: Path, target_size: int,
                dry_run: bool) -> list[dict]:
    """Write images and masks for one split. Returns per-record stats."""
    img_dir  = out_root / split_name / "images"
    mask_dir = out_root / split_name / "masks"
    if not dry_run:
        img_dir.mkdir(parents=True, exist_ok=True)
        mask_dir.mkdir(parents=True, exist_ok=True)

    written = 0
    skipped = 0
    stats   = []

    for i, rec in enumerate(records):
        if dry_run:
            written += 1
            continue

        img, mask = generate_mask(rec, target_size)
        if img is None:
            skipped += 1
            continue

        stem = f"{split_name}_{i:05d}"
        cv2.imwrite(str(img_dir  / f"{stem}.jpg"), img,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        cv2.imwrite(str(mask_dir / f"{stem}.png"), mask)

        sev = compute_severity(mask)
        stats.append({
            "file":         stem,
            "source":       rec["source"],
            "labels":       list({b["label"] for b in rec["boxes"]}),
            "num_boxes":    len(rec["boxes"]),
            "infected_pct": sev["infected_pct"],
            "severity":     sev["severity"],
        })
        written += 1

    print(f"  {split_name:5s}: {written} pairs written  "
          f"({skipped} skipped)" + (" [DRY RUN]" if dry_run else ""))
    return stats


# ══════════════════════════════════════════════════════════════════════════════
# ❻  REPORT
# ══════════════════════════════════════════════════════════════════════════════

def print_report(all_stats: dict, total: int):
    from collections import Counter

    print("\n" + "═" * 65)
    print("  STAGE 2 DATASET REPORT")
    print("═" * 65)

    for split_name, stats in all_stats.items():
        if not stats:
            continue
        sev_counts = Counter(s["severity"] for s in stats)
        src_counts = Counter(s["source"]   for s in stats)
        avg_pct    = sum(s["infected_pct"] for s in stats) / len(stats)

        print(f"\n  {split_name.upper()}  ({len(stats)} images)")
        print(f"  {'Severity':<12}  Count")
        for sev in ["Healthy", "Mild", "Moderate", "Severe"]:
            n   = sev_counts.get(sev, 0)
            bar = "▓" * (n * 20 // max(len(stats), 1))
            print(f"    {sev:<10}  {n:>5}  {bar}")
        print(f"  Avg infected area : {avg_pct:.1f}%")
        print(f"  Sources           : {dict(src_counts)}")

    print(f"\n  TOTAL pairs: {total}")
    print("═" * 65)


# ══════════════════════════════════════════════════════════════════════════════
# ❼  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main(dry_run: bool = False):
    random.seed(SEED)
    print("=" * 65)
    print("  Stage 2 — Dataset Preprocessing")
    print("=" * 65)
    if dry_run:
        print("  ▶️  DRY RUN — no files will be written\n")

    # ── Collect ───────────────────────────────────────────────────────────────
    print("\nCollecting annotations...")
    records = []
    records += collect_ccl20(CCL20_IMAGES_DIR, CCL20_ANNOTATIONS_DIR)
    records += collect_yolo(YOLO_IMG_DIR, YOLO_TXT_DIR)

    # Deduplicate by resolved absolute path
    seen, unique = set(), []
    for r in records:
        p = r["image_path"].resolve()
        if p not in seen:
            seen.add(p)
            unique.append(r)

    print(f"\n  Total unique annotated images : {len(unique)}")
    print(f"  Total bounding boxes          : "
          f"{sum(len(r['boxes']) for r in unique)}")

    if not unique:
        print("\n[ERROR] No annotated images found. Check your paths above.")
        return

    # ── Split ─────────────────────────────────────────────────────────────────
    random.shuffle(unique)
    n  = len(unique)
    t1 = int(n * SPLIT[0])
    t2 = t1 + int(n * SPLIT[1])
    splits = {
        "train": unique[:t1],
        "val":   unique[t1:t2],
        "test":  unique[t2:],
    }

    print(f"\n  Split: train={len(splits['train'])}  "
          f"val={len(splits['val'])}  test={len(splits['test'])}")

    # ── Write ─────────────────────────────────────────────────────────────────
    out_root = Path(OUTPUT_DIR)
    if not dry_run and out_root.exists():
        shutil.rmtree(out_root)

    print(f"\nGenerating masks → {OUTPUT_DIR}/")
    print(f"  Target image size : {TARGET_SIZE}×{TARGET_SIZE}")

    all_stats: dict[str, list] = {}
    total_written = 0
    for split_name, recs in splits.items():
        stats = write_split(recs, split_name, out_root, TARGET_SIZE, dry_run)
        all_stats[split_name] = stats
        total_written += len(stats)

    # ── Save info JSON ────────────────────────────────────────────────────────
    if not dry_run:
        info = {
            "target_size": TARGET_SIZE,
            "split_ratio": list(SPLIT),
            "total":       total_written,
            "splits":      {k: len(v) for k, v in all_stats.items()},
            "records":     {k: v for k, v in all_stats.items()},
        }
        info_path = out_root / "dataset_info.json"
        with open(info_path, "w") as f:
            json.dump(info, f, indent=2)
        print(f"\n  ✓ Dataset info saved → {info_path}")

    # ── Report ────────────────────────────────────────────────────────────────
    if not dry_run:
        print_report(all_stats, total_written)
        print(f"\n✓ Stage 2 dataset ready at ./{OUTPUT_DIR}/")
        print("  Structure:")
        print("    stage2_dataset/")
        print("      train/images/  ← 512×512 JPG")
        print("      train/masks/   ← 512×512 PNG  (white=infected)")
        print("      val/images/  val/masks/")
        print("      test/images/ test/masks/")
        print("      dataset_info.json")
    else:
        print(f"\n  Dry run complete. Would write {total_written} image+mask pairs.")
        print("  Run without --dry-run to generate the actual files.")


main(dry_run=False)

  Stage 2 — Dataset Preprocessing

  CCL20 XML  → 1817 annotated images  (8331 total boxes)
  YOLO classes: ['Anthracnose', 'Bacterial Blight', 'Citrus Canker', 'Curl Virus', 'Deficiency Leaf', 'Dry Leaf', 'Healthy Leaf', 'Sooty Mould', 'Spider Mites']
  YOLO TXT   → 900 annotated images  (901 total boxes)

  Total unique annotated images : 2717
  Total bounding boxes          : 9232

  Split: train=1901  val=407  test=409

Generating masks → stage2_dataset/
  Target image size : 512×512
  train: 1901 pairs written  (0 skipped)
  val  : 407 pairs written  (0 skipped)
  test : 409 pairs written  (0 skipped)

  ✓ Dataset info saved → stage2_dataset/dataset_info.json

═════════════════════════════════════════════════════════════════
  STAGE 2 DATASET REPORT
═════════════════════════════════════════════════════════════════

  TRAIN  (1901 images)
  Severity      Count
    Healthy       488  ▓▓▓▓▓
    Mild          365  ▓▓▓
    Moderate      361  ▓▓▓
    Severe        687  ▓▓▓▓▓▓▓
  Avg inf

In [ ]:
"""
Stage 2 — Hybrid Severity Model
=================================
Formula:
  Severity Score = 0.4 × Area + 0.25 × Color Deviation
                 + 0.20 × Spot Spread + 0.15 × Zone Distribution

Four signals:
  1. Area Score       — % of leaf pixels that are infected
  2. Color Deviation  — how far the leaf deviates from healthy green (HSV/LAB)
  3. Spot Spread      — number of spots + how scattered they are
  4. Zone Score       — which zones of the leaf are infected (spatial map)
                        Leaf divided into 3x3 = 9 zones.
                        More zones infected = higher systemic spread score.

Output:
  {
    "disease":         "Canker",
    "confidence":      "94.2%",
    "severity_score":  68.4,
    "severity_label":  "Moderate",
    "breakdown": {
      "area_score":    50.2,
      "color_score":   20.1,
      "spread_score":  30.5,
      "zone_score":    45.0
    },
    "spot_count":      7,
    "zones_infected":  5,
    "infected_pct":    42.3
  }

Usage:
  python stage2_hybrid_severity.py --image path/to/leaf.jpg
  python stage2_hybrid_severity.py --batch path/to/folder/
  python stage2_hybrid_severity.py --evaluate   # run on test set + save report
"""

import os
import cv2
import json
import argparse
import numpy as np
from pathlib import Path
from dataclasses import dataclass

# ══════════════════════════════════════════════════════════════════════════════
# ❶  CONFIG
# ══════════════════════════════════════════════════════════════════════════════

# Severity thresholds (final weighted score 0–100)
SEVERITY_THRESHOLDS = {
    "Mild":     (0,   25),
    "Moderate": (25,  50),
    "Severe":   (50,  75),
    "Critical": (75,  100),
}

# Formula weights (must sum to 1.0)
W_AREA   = 0.40
W_COLOR  = 0.25
W_SPREAD = 0.20
W_ZONE   = 0.15

# Healthy green reference in LAB color space
# L=128, A=110, B=145 roughly corresponds to healthy leaf green
HEALTHY_LAB = np.array([128, 110, 145], dtype=np.float32)

# GrabCut / HSV fallback for leaf segmentation
HSV_LOWER_GREEN = np.array([25,  30,  30])   # lower bound for leaf green
HSV_UPPER_GREEN = np.array([95, 255, 255])   # upper bound for leaf green

# ══════════════════════════════════════════════════════════════════════════════
# ❷  LEAF SEGMENTATION  (isolate leaf from background)
# ══════════════════════════════════════════════════════════════════════════════

def segment_leaf(img_bgr: np.ndarray) -> np.ndarray:
    """
    Returns a binary mask (H, W) where 255 = leaf, 0 = background.
    Uses HSV thresholding to find the green leaf region.
    Falls back to full image if leaf mask is too small.
    """
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask    = cv2.inRange(img_hsv, HSV_LOWER_GREEN, HSV_UPPER_GREEN)

    # Morphological cleanup — remove noise, fill holes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=2)

    # If detected leaf area < 10% of image, fall back to full image
    leaf_ratio = mask.sum() / (mask.size * 255)
    if leaf_ratio < 0.10:
        mask = np.ones(img_bgr.shape[:2], dtype=np.uint8) * 255

    return mask


# ══════════════════════════════════════════════════════════════════════════════
# ❸  INFECTED REGION DETECTION  (find diseased pixels)
# ══════════════════════════════════════════════════════════════════════════════

def detect_infected_region(img_bgr: np.ndarray,
                            leaf_mask: np.ndarray) -> np.ndarray:
    """
    Returns binary mask (H, W) where 255 = infected pixel.

    Strategy:
      In HSV space, infected regions (yellow, brown, dark spots) fall
      outside the healthy green range. Within the leaf mask, pixels that
      are NOT green are considered infected.
    """
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # Healthy green range (same as leaf segmentation)
    healthy_mask = cv2.inRange(img_hsv, HSV_LOWER_GREEN, HSV_UPPER_GREEN)

    # Infected = leaf pixels that are not healthy green
    infected = cv2.bitwise_and(
        cv2.bitwise_not(healthy_mask),
        leaf_mask
    )

    # Morphological cleanup
    kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    infected = cv2.morphologyEx(infected, cv2.MORPH_CLOSE, kernel, iterations=2)
    infected = cv2.morphologyEx(infected, cv2.MORPH_OPEN,  kernel, iterations=1)

    return infected


# ══════════════════════════════════════════════════════════════════════════════
# ❹  SIGNAL 1 — AREA SCORE
# ══════════════════════════════════════════════════════════════════════════════

def compute_area_score(infected_mask: np.ndarray,
                        leaf_mask: np.ndarray) -> float:
    """
    area_score = (infected_pixels / leaf_pixels) × 100
    Returns 0–100.
    """
    leaf_pixels     = max(1, (leaf_mask > 0).sum())
    infected_pixels = (infected_mask > 0).sum()
    return min(100.0, float(infected_pixels / leaf_pixels * 100))


# ══════════════════════════════════════════════════════════════════════════════
# ❺  SIGNAL 2 — COLOR DEVIATION SCORE
# ══════════════════════════════════════════════════════════════════════════════

def compute_color_score(img_bgr: np.ndarray,
                         leaf_mask: np.ndarray) -> float:
    """
    Measures how far the average leaf color deviates from healthy green
    in LAB color space (perceptually uniform — distances mean the same
    thing across the color spectrum).

    color_score = (mean_delta_E / max_delta_E) × 100
    Returns 0–100.

    Delta E is the Euclidean distance in LAB space between each pixel
    and the healthy green reference color.
    """
    img_lab  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)

    # Extract only leaf pixels
    leaf_pixels = img_lab[leaf_mask > 0]   # shape (N, 3)
    if len(leaf_pixels) == 0:
        return 0.0

    # Delta E per pixel (Euclidean distance in LAB)
    delta_e     = np.linalg.norm(leaf_pixels - HEALTHY_LAB, axis=1)
    mean_delta  = delta_e.mean()

    # Normalise: max possible LAB distance ≈ 260
    MAX_DELTA_E = 260.0
    score       = min(100.0, float(mean_delta / MAX_DELTA_E * 100))
    return score


# ══════════════════════════════════════════════════════════════════════════════
# ❻  SIGNAL 3 — SPOT SPREAD SCORE  (the novel component)
# ══════════════════════════════════════════════════════════════════════════════

def compute_spread_score(infected_mask: np.ndarray) -> tuple[float, int]:
    """
    Finds individual disease spots using contour detection.
    Computes:
      - num_spots   : number of distinct infected regions
      - spread      : average pairwise distance between spot centroids

    spread_score = normalised combination of count + spread distance
    Returns (score 0–100, num_spots).

    Why this matters:
      5 tightly clustered spots = less severe than 5 widely spread spots
      even if the infected area is the same.
      Spread indicates systemic infection spreading across the leaf.
    """
    contours, _ = cv2.findContours(
        infected_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    # Filter tiny contours (noise) — min area = 0.01% of image
    min_area = infected_mask.size * 0.0001
    contours = [c for c in contours if cv2.contourArea(c) > min_area]
    num_spots = len(contours)

    if num_spots == 0:
        return 0.0, 0

    # Compute centroid of each spot
    centroids = []
    for c in contours:
        M = cv2.moments(c)
        if M["m00"] > 0:
            cx = M["m10"] / M["m00"]
            cy = M["m01"] / M["m00"]
            centroids.append((cx, cy))

    # Average pairwise distance between centroids
    if len(centroids) < 2:
        avg_dist = 0.0
    else:
        distances = []
        for i in range(len(centroids)):
            for j in range(i + 1, len(centroids)):
                dx = centroids[i][0] - centroids[j][0]
                dy = centroids[i][1] - centroids[j][1]
                distances.append(np.sqrt(dx**2 + dy**2))
        avg_dist = np.mean(distances)

    # Normalise:
    #   spot count: cap at 20 spots
    #   distance:   cap at half the image diagonal
    h, w     = infected_mask.shape
    max_dist = np.sqrt(h**2 + w**2) / 2

    count_score = min(1.0, num_spots / 20.0)
    dist_score  = min(1.0, avg_dist  / max_dist) if max_dist > 0 else 0.0

    # Equal weight to count and spread distance
    spread_score = (0.5 * count_score + 0.5 * dist_score) * 100
    return float(spread_score), num_spots


# ══════════════════════════════════════════════════════════════════════════════
# ❼  SIGNAL 4 — ZONE DISTRIBUTION SCORE  (spatial component)
# ══════════════════════════════════════════════════════════════════════════════

def compute_zone_score(infected_mask: np.ndarray,
                        grid: int = 3) -> tuple[float, int]:
    """
    Divides the leaf image into a grid×grid spatial grid (default 3×3 = 9 zones).
    Checks which zones contain infected pixels.

    zone_score = (zones_infected / total_zones) × 100
    Returns (score 0–100, zones_infected count).

    Why this matters:
      Area score tells you HOW MUCH is infected.
      Zone score tells you WHERE — is it confined to one corner
      or spread across the whole leaf?

      Example:
        Image A: 30% infected, all in one zone    → zone_score = 11%
        Image B: 30% infected, across 7 zones     → zone_score = 78%
      Image B is clinically more severe — systemic infection.

    Viva defence:
      "We divide the leaf into a 3×3 spatial grid and count how many
       zones contain infected tissue. A localised infection scores low
       even if the infected area is large; a multi-zone infection scores
       high because it indicates systemic spread across the leaf surface."
    """
    h, w        = infected_mask.shape
    total_zones = grid * grid
    zone_h      = h // grid
    zone_w      = w // grid
    zones_infected = 0

    for row in range(grid):
        for col in range(grid):
            y1 = row * zone_h
            y2 = y1 + zone_h if row < grid - 1 else h
            x1 = col * zone_w
            x2 = x1 + zone_w if col < grid - 1 else w
            zone = infected_mask[y1:y2, x1:x2]
            # Zone counts as infected if > 1% of its pixels are infected
            if zone.size > 0 and (zone > 0).mean() > 0.01:
                zones_infected += 1

    zone_score = (zones_infected / total_zones) * 100
    return float(zone_score), zones_infected


# ══════════════════════════════════════════════════════════════════════════════
# ❽  FINAL SEVERITY COMPUTATION
# ══════════════════════════════════════════════════════════════════════════════

def severity_label(score: float) -> str:
    for label, (lo, hi) in SEVERITY_THRESHOLDS.items():
        if lo <= score < hi:
            return label
    return "Critical"


def compute_severity(img_bgr: np.ndarray) -> dict:
    """
    Main function. Takes a BGR image array, returns full severity breakdown.
    """
    # Step 1: segment leaf from background
    leaf_mask = segment_leaf(img_bgr)

    # Step 2: detect infected regions within leaf
    infected_mask = detect_infected_region(img_bgr, leaf_mask)

    # Step 3: compute four signals
    area_score              = compute_area_score(infected_mask, leaf_mask)
    color_score             = compute_color_score(img_bgr, leaf_mask)
    spread_score, num_spots = compute_spread_score(infected_mask)
    zone_score, zones_infected = compute_zone_score(infected_mask)

    # Step 4: weighted combination
    final_score = (W_AREA   * area_score   +
                   W_COLOR  * color_score  +
                   W_SPREAD * spread_score +
                   W_ZONE   * zone_score)

    infected_pct = area_score   # area score IS the infected percentage

    return {
        "severity_score":  round(final_score,   2),
        "severity_label":  severity_label(final_score),
        "infected_pct":    round(infected_pct,  2),
        "spot_count":      num_spots,
        "zones_infected":  zones_infected,
        "breakdown": {
            "area_score":   round(area_score,   2),
            "color_score":  round(color_score,  2),
            "spread_score": round(spread_score, 2),
            "zone_score":   round(zone_score,   2),
        },
    }


# ══════════════════════════════════════════════════════════════════════════════
# ❽  VISUALISATION  (for demo + report)
# ══════════════════════════════════════════════════════════════════════════════

def visualise_severity(img_bgr: np.ndarray,
                        result: dict,
                        disease: str = "",
                        save_path: str = None) -> np.ndarray:
    """
    Creates a side-by-side panel:
      Original | Leaf mask | Infected overlay | Score breakdown

    Returns the composite image (BGR numpy array).
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    h, w  = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Recompute masks for visualisation
    leaf_mask     = segment_leaf(img_bgr)
    infected_mask = detect_infected_region(img_bgr, leaf_mask)

    # Overlay: green = healthy leaf, red = infected
    overlay = img_rgb.copy()
    overlay[leaf_mask     > 0] = (overlay[leaf_mask     > 0] * 0.6 +
                                   np.array([0, 180, 0])     * 0.4).astype(np.uint8)
    overlay[infected_mask > 0] = (overlay[infected_mask > 0] * 0.4 +
                                   np.array([220, 50, 50])   * 0.6).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor("#1e1e1e")

    axes[0].imshow(img_rgb)
    axes[0].set_title("Original", color="white", fontsize=12)
    axes[0].axis("off")

    axes[1].imshow(overlay)
    green_patch = mpatches.Patch(color=(0, 180/255, 0), label="Healthy tissue")
    red_patch   = mpatches.Patch(color=(220/255, 50/255, 50/255), label="Infected region")
    axes[1].legend(handles=[green_patch, red_patch],
                   loc="lower left", fontsize=9,
                   facecolor="#333", labelcolor="white")
    axes[1].set_title("Segmentation overlay", color="white", fontsize=12)
    axes[1].axis("off")

    # Score breakdown bar chart
    bd      = result["breakdown"]
    labels  = ["Area\n(×0.5)", "Color\n(×0.3)", "Spread\n(×0.2)"]
    raw     = [bd["area_score"], bd["color_score"], bd["spread_score"]]
    contrib = [bd["area_score"]  * W_AREA,
               bd["color_score"] * W_COLOR,
               bd["spread_score"]* W_SPREAD]
    colors  = ["#E07B54", "#E0C454", "#54A0E0"]

    ax = axes[2]
    ax.set_facecolor("#2a2a2a")
    bars = ax.bar(labels, contrib, color=colors, edgecolor="white",
                  linewidth=0.5, width=0.5)
    for bar, raw_val in zip(bars, raw):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f"{raw_val:.1f}",
                ha="center", va="bottom", color="white", fontsize=10)

    ax.set_ylim(0, 55)
    ax.set_ylabel("Weighted contribution", color="white")
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#555")

    label  = result["severity_label"]
    score  = result["severity_score"]
    color_map = {"Mild": "#54C054", "Moderate": "#E0C454",
                 "Severe": "#E07B54", "Critical": "#CC3333"}
    ax.set_title(
        f"{disease}  |  {label}  ({score:.1f}%)\n"
        f"Infected: {result['infected_pct']:.1f}%  •  "
        f"Spots: {result['spot_count']}",
        color=color_map.get(label, "white"), fontsize=11, fontweight="bold"
    )

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight",
                    facecolor="#1e1e1e")
        print(f"  ✓ Visualisation saved → {save_path}")
    plt.show()
    plt.close()

    return overlay


# ══════════════════════════════════════════════════════════════════════════════
# ❾  STAGE 1 + STAGE 2 COMBINED PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def load_stage1_model():
    """Load EfficientNet-B0 classifier from Stage 1."""
    import torch
    import torch.nn as nn
    from torchvision import models, transforms
    from PIL import Image as PILImage

    MODEL_PATH = "models/classifier.pth"
    CLASS_MAP  = "models/class_map.json"

    if not Path(MODEL_PATH).exists():
        print("  [WARN] Stage 1 model not found — skipping disease prediction")
        return None, None, None

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    with open(CLASS_MAP) as f:
        cmap = json.load(f)
    idx_to_class = {int(k): v for k, v in cmap["idx_to_class"].items()}
    num_classes  = len(idx_to_class)

    model = models.efficientnet_b0(weights=None)
    in_f  = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, num_classes))
    ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval().to(DEVICE)

    tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                              [0.229, 0.224, 0.225]),
    ])
    return model, idx_to_class, tf


def predict_full(image_path: str, save_viz: bool = False) -> dict:
    """
    Full pipeline: Stage 1 (disease) + Stage 2 (severity).
    """
    import torch
    from PIL import Image as PILImage

    # Load image
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read: {image_path}")

    result = {"image": image_path}

    # ── Stage 1: disease classification ──────────────────────────────────────
    s1_model, idx_to_class, tf = load_stage1_model()
    if s1_model is not None:
        DEVICE = next(s1_model.parameters()).device
        pil_img = PILImage.open(image_path).convert("RGB")
        inp     = tf(pil_img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs      = torch.softmax(s1_model(inp), dim=1)[0]
            conf, idx  = probs.max(0)
        result["disease"]    = idx_to_class[idx.item()]
        result["confidence"] = f"{conf.item()*100:.1f}%"
    else:
        result["disease"]    = "Unknown"
        result["confidence"] = "N/A"

    # ── Stage 2: severity ─────────────────────────────────────────────────────
    # Skip severity if leaf is healthy
    if result["disease"].lower() == "healthy":
        result.update({
            "severity_score": 0.0,
            "severity_label": "Healthy",
            "infected_pct":   0.0,
            "spot_count":     0,
            "breakdown": {"area_score": 0.0,
                          "color_score": 0.0,
                          "spread_score": 0.0},
        })
    else:
        severity = compute_severity(img_bgr)
        result.update(severity)

    # ── Visualise ─────────────────────────────────────────────────────────────
    if save_viz:
        output_dir = Path("stage2_severity_output")
        output_dir.mkdir(exist_ok=True)
        viz_path = output_dir / (Path(image_path).stem + "_severity.png")
        visualise_severity(img_bgr, result,
                           disease=result["disease"],
                           save_path=str(viz_path))

    return result


# ══════════════════════════════════════════════════════════════════════════════
# ❿  BATCH EVALUATION on stage2_dataset/test/
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_on_test_set(test_dir: str = "stage2_dataset/test",
                          output_json: str = "models/stage2_eval_results.json"):
    """
    Runs severity on all test images, saves results + severity distribution.
    """
    from collections import Counter

    test_path = Path(test_dir) / "images"
    if not test_path.exists():
        print(f"[ERROR] Test dir not found: {test_path}")
        return

    img_files = sorted(test_path.glob("*.jpg"))
    print(f"Evaluating severity on {len(img_files)} test images...")

    results  = []
    sev_dist = Counter()

    for i, img_path in enumerate(img_files):
        img_bgr  = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        sev = compute_severity(img_bgr)
        sev["image"] = img_path.name
        results.append(sev)
        sev_dist[sev["severity_label"]] += 1

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(img_files)}] done...")

    os.makedirs("models", exist_ok=True)
    with open(output_json, "w") as f:
        json.dump(results, f, indent=2)

    # Summary
    print(f"\n{'═'*55}")
    print(f"  STAGE 2 EVALUATION — {len(results)} test images")
    print(f"{'═'*55}")
    avg_score = np.mean([r["severity_score"] for r in results])
    avg_area  = np.mean([r["infected_pct"]   for r in results])
    avg_spots = np.mean([r["spot_count"]     for r in results])
    print(f"  Avg severity score : {avg_score:.1f}")
    print(f"  Avg infected area  : {avg_area:.1f}%")
    print(f"  Avg spot count     : {avg_spots:.1f}")
    print(f"\n  Severity distribution:")
    for sev in ["Mild", "Moderate", "Severe", "Critical"]:
        n   = sev_dist.get(sev, 0)
        pct = n / len(results) * 100 if results else 0
        bar = "▓" * int(pct / 3)
        print(f"    {sev:<10}: {n:>4}  ({pct:.1f}%)  {bar}")
    print(f"\n  ✓ Results saved → {output_json}")
    print(f"{'═'*55}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # Modified for Jupyter notebook: directly process stage2_dataset/train
    batch_folder = "stage2_dataset/train/images"
    save_viz = True  # Set to False if you don't want visualizations
    
    results = []
    for img_path in sorted(Path(batch_folder).glob("*.jpg")):
        r = predict_full(str(img_path), save_viz=save_viz)
        print(f"  {img_path.name:30s} → "
              f"{r.get('disease','?'):15s} | "
              f"{r['severity_label']:10s} "
              f"({r['severity_score']:.1f}%)")
        results.append(r)
    with open("batch_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"\n✓ Saved batch_results.json")

  ✓ Visualisation saved → train_00000_severity.png
  train_00000.jpg                → DeficiencyLeaf  | Moderate   (26.1%)


/var/folders/fq/fn55_cv53qqfd0n46tbgzj800000gn/T/ipykernel_77561/4198502936.py:436: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  ✓ Visualisation saved → train_00001_severity.png
  train_00001.jpg                → Anthracnose     | Severe     (62.7%)
  ✓ Visualisation saved → train_00002_severity.png
  train_00002.jpg                → LeafMiner       | Mild       (22.0%)
  ✓ Visualisation saved → train_00003_severity.png
  train_00003.jpg                → DeficiencyLeaf  | Severe     (63.6%)
  ✓ Visualisation saved → train_00004_severity.png
  train_00004.jpg                → BlackSpot       | Mild       (24.1%)
  ✓ Visualisation saved → train_00005_severity.png
  train_00005.jpg                → SpiderMites     | Mild       (21.0%)
  ✓ Visualisation saved → train_00006_severity.png
  train_00006.jpg                → Greening        | Mild       (16.2%)
  ✓ Visualisation saved → train_00007_severity.png
  train_00007.jpg                → SpiderMites     | Moderate   (43.5%)
  ✓ Visualisation saved → train_00008_severity.png
  train_00008.jpg                → CurlVirus       | Moderate   (43.7%)
  ✓ Visualisatio

In [4]:
import shutil
from pathlib import Path

# Move existing severity images to the output folder
output_dir = Path("stage2_severity_output")
output_dir.mkdir(exist_ok=True)

moved_count = 0
for f in Path(".").glob("train_*_severity.png"):
    shutil.move(str(f), str(output_dir / f.name))
    moved_count += 1

print(f"Moved {moved_count} severity images to stage2_severity_output/")

Moved 1901 severity images to stage2_severity_output/


Second try

In [1]:
"""
Stage 2 — Hybrid Severity Model
=================================
Formula:
  Severity Score = 0.4 × Area + 0.25 × Color Deviation
                 + 0.20 × Spot Spread + 0.15 × Zone Distribution

Four signals:
  1. Area Score       — % of leaf pixels that are infected
  2. Color Deviation  — how far the leaf deviates from healthy green (HSV/LAB)
  3. Spot Spread      — number of spots + how scattered they are
  4. Zone Score       — which zones of the leaf are infected (spatial map)
                        Leaf divided into 3x3 = 9 zones.
                        More zones infected = higher systemic spread score.

Output:
  {
    "disease":         "Canker",
    "confidence":      "94.2%",
    "severity_score":  68.4,
    "severity_label":  "Moderate",
    "breakdown": {
      "area_score":    50.2,
      "color_score":   20.1,
      "spread_score":  30.5,
      "zone_score":    45.0
    },
    "spot_count":      7,
    "zones_infected":  5,
    "infected_pct":    42.3
  }

Usage:
  python stage2_hybrid_severity.py --image path/to/leaf.jpg
  python stage2_hybrid_severity.py --batch path/to/folder/
  python stage2_hybrid_severity.py --evaluate   # run on test set + save report
"""

import os
import cv2
import json
import argparse
import numpy as np
from pathlib import Path
from dataclasses import dataclass

# ══════════════════════════════════════════════════════════════════════════════
# ❶  CONFIG
# ══════════════════════════════════════════════════════════════════════════════

# Severity thresholds (final weighted score 0–100)
SEVERITY_THRESHOLDS = {
    "Mild":     (0,   25),
    "Moderate": (25,  50),
    "Severe":   (50,  75),
    "Critical": (75,  100),
}

# Formula weights (must sum to 1.0)
W_AREA   = 0.40
W_COLOR  = 0.25
W_SPREAD = 0.20
W_ZONE   = 0.15

# Healthy green reference in LAB color space
# L=128, A=110, B=145 roughly corresponds to healthy leaf green
HEALTHY_LAB = np.array([128, 110, 145], dtype=np.float32)

# GrabCut / HSV fallback for leaf segmentation
HSV_LOWER_GREEN = np.array([25,  30,  30])   # lower bound for leaf green
HSV_UPPER_GREEN = np.array([95, 255, 255])   # upper bound for leaf green

# ══════════════════════════════════════════════════════════════════════════════
# ❷  LEAF SEGMENTATION  (isolate leaf from background)
# ══════════════════════════════════════════════════════════════════════════════

def segment_leaf(img_bgr: np.ndarray) -> np.ndarray:
    """
    Returns a binary mask (H, W) where 255 = leaf, 0 = background.
    Uses HSV thresholding to find the green leaf region.
    Falls back to full image if leaf mask is too small.
    """
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask    = cv2.inRange(img_hsv, HSV_LOWER_GREEN, HSV_UPPER_GREEN)

    # Morphological cleanup — remove noise, fill holes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=2)

    # If detected leaf area < 10% of image, fall back to full image
    leaf_ratio = mask.sum() / (mask.size * 255)
    if leaf_ratio < 0.10:
        mask = np.ones(img_bgr.shape[:2], dtype=np.uint8) * 255

    return mask


# ══════════════════════════════════════════════════════════════════════════════
# ❸  INFECTED REGION DETECTION  (find diseased pixels)
# ══════════════════════════════════════════════════════════════════════════════

def detect_infected_region(img_bgr: np.ndarray,
                            leaf_mask: np.ndarray) -> np.ndarray:
    """
    Returns binary mask (H, W) where 255 = infected pixel.

    Strategy:
      In HSV space, infected regions (yellow, brown, dark spots) fall
      outside the healthy green range. Within the leaf mask, pixels that
      are NOT green are considered infected.
    """
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # Healthy green range (same as leaf segmentation)
    healthy_mask = cv2.inRange(img_hsv, HSV_LOWER_GREEN, HSV_UPPER_GREEN)

    # Infected = leaf pixels that are not healthy green
    infected = cv2.bitwise_and(
        cv2.bitwise_not(healthy_mask),
        leaf_mask
    )

    # Morphological cleanup
    kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    infected = cv2.morphologyEx(infected, cv2.MORPH_CLOSE, kernel, iterations=2)
    infected = cv2.morphologyEx(infected, cv2.MORPH_OPEN,  kernel, iterations=1)

    return infected


# ══════════════════════════════════════════════════════════════════════════════
# ❹  SIGNAL 1 — AREA SCORE
# ══════════════════════════════════════════════════════════════════════════════

def compute_area_score(infected_mask: np.ndarray,
                        leaf_mask: np.ndarray) -> float:
    """
    area_score = (infected_pixels / leaf_pixels) × 100
    Returns 0–100.
    """
    leaf_pixels     = max(1, (leaf_mask > 0).sum())
    infected_pixels = (infected_mask > 0).sum()
    return min(100.0, float(infected_pixels / leaf_pixels * 100))


# ══════════════════════════════════════════════════════════════════════════════
# ❺  SIGNAL 2 — COLOR DEVIATION SCORE
# ══════════════════════════════════════════════════════════════════════════════

def compute_color_score(img_bgr: np.ndarray,
                         leaf_mask: np.ndarray) -> float:
    """
    Measures how far the average leaf color deviates from healthy green
    in LAB color space (perceptually uniform — distances mean the same
    thing across the color spectrum).

    color_score = (mean_delta_E / max_delta_E) × 100
    Returns 0–100.

    Delta E is the Euclidean distance in LAB space between each pixel
    and the healthy green reference color.
    """
    img_lab  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)

    # Extract only leaf pixels
    leaf_pixels = img_lab[leaf_mask > 0]   # shape (N, 3)
    if len(leaf_pixels) == 0:
        return 0.0

    # Delta E per pixel (Euclidean distance in LAB)
    delta_e     = np.linalg.norm(leaf_pixels - HEALTHY_LAB, axis=1)
    mean_delta  = delta_e.mean()

    # Normalise: max possible LAB distance ≈ 260
    MAX_DELTA_E = 260.0
    score       = min(100.0, float(mean_delta / MAX_DELTA_E * 100))
    return score


# ══════════════════════════════════════════════════════════════════════════════
# ❻  SIGNAL 3 — SPOT SPREAD SCORE  (the novel component)
# ══════════════════════════════════════════════════════════════════════════════

def compute_spread_score(infected_mask: np.ndarray) -> tuple[float, int]:
    """
    Finds individual disease spots using contour detection.
    Computes:
      - num_spots   : number of distinct infected regions
      - spread      : average pairwise distance between spot centroids

    spread_score = normalised combination of count + spread distance
    Returns (score 0–100, num_spots).

    Why this matters:
      5 tightly clustered spots = less severe than 5 widely spread spots
      even if the infected area is the same.
      Spread indicates systemic infection spreading across the leaf.
    """
    contours, _ = cv2.findContours(
        infected_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    # Filter tiny contours (noise) — min area = 0.01% of image
    min_area = infected_mask.size * 0.0001
    contours = [c for c in contours if cv2.contourArea(c) > min_area]
    num_spots = len(contours)

    if num_spots == 0:
        return 0.0, 0

    # Compute centroid of each spot
    centroids = []
    for c in contours:
        M = cv2.moments(c)
        if M["m00"] > 0:
            cx = M["m10"] / M["m00"]
            cy = M["m01"] / M["m00"]
            centroids.append((cx, cy))

    # Average pairwise distance between centroids
    if len(centroids) < 2:
        avg_dist = 0.0
    else:
        distances = []
        for i in range(len(centroids)):
            for j in range(i + 1, len(centroids)):
                dx = centroids[i][0] - centroids[j][0]
                dy = centroids[i][1] - centroids[j][1]
                distances.append(np.sqrt(dx**2 + dy**2))
        avg_dist = np.mean(distances)

    # Normalise:
    #   spot count: cap at 20 spots
    #   distance:   cap at half the image diagonal
    h, w     = infected_mask.shape
    max_dist = np.sqrt(h**2 + w**2) / 2

    count_score = min(1.0, num_spots / 20.0)
    dist_score  = min(1.0, avg_dist  / max_dist) if max_dist > 0 else 0.0

    # Equal weight to count and spread distance
    spread_score = (0.5 * count_score + 0.5 * dist_score) * 100
    return float(spread_score), num_spots


# ══════════════════════════════════════════════════════════════════════════════
# ❼  SIGNAL 4 — ZONE DISTRIBUTION SCORE  (spatial component)
# ══════════════════════════════════════════════════════════════════════════════

def compute_zone_score(infected_mask: np.ndarray,
                        grid: int = 3) -> tuple[float, int]:
    """
    Divides the leaf image into a grid×grid spatial grid (default 3×3 = 9 zones).
    Checks which zones contain infected pixels.

    zone_score = (zones_infected / total_zones) × 100
    Returns (score 0–100, zones_infected count).

    Why this matters:
      Area score tells you HOW MUCH is infected.
      Zone score tells you WHERE — is it confined to one corner
      or spread across the whole leaf?

      Example:
        Image A: 30% infected, all in one zone    → zone_score = 11%
        Image B: 30% infected, across 7 zones     → zone_score = 78%
      Image B is clinically more severe — systemic infection.

    Viva defence:
      "We divide the leaf into a 3×3 spatial grid and count how many
       zones contain infected tissue. A localised infection scores low
       even if the infected area is large; a multi-zone infection scores
       high because it indicates systemic spread across the leaf surface."
    """
    h, w        = infected_mask.shape
    total_zones = grid * grid
    zone_h      = h // grid
    zone_w      = w // grid
    zones_infected = 0

    for row in range(grid):
        for col in range(grid):
            y1 = row * zone_h
            y2 = y1 + zone_h if row < grid - 1 else h
            x1 = col * zone_w
            x2 = x1 + zone_w if col < grid - 1 else w
            zone = infected_mask[y1:y2, x1:x2]
            # Zone counts as infected if > 1% of its pixels are infected
            if zone.size > 0 and (zone > 0).mean() > 0.01:
                zones_infected += 1

    zone_score = (zones_infected / total_zones) * 100
    return float(zone_score), zones_infected


# ══════════════════════════════════════════════════════════════════════════════
# ❽  FINAL SEVERITY COMPUTATION
# ══════════════════════════════════════════════════════════════════════════════

def severity_label(score: float) -> str:
    for label, (lo, hi) in SEVERITY_THRESHOLDS.items():
        if lo <= score < hi:
            return label
    return "Critical"


def compute_severity(img_bgr: np.ndarray) -> dict:
    """
    Main function. Takes a BGR image array, returns full severity breakdown.
    """
    # Step 1: segment leaf from background
    leaf_mask = segment_leaf(img_bgr)

    # Step 2: detect infected regions within leaf
    infected_mask = detect_infected_region(img_bgr, leaf_mask)

    # Step 3: compute four signals
    area_score              = compute_area_score(infected_mask, leaf_mask)
    color_score             = compute_color_score(img_bgr, leaf_mask)
    spread_score, num_spots = compute_spread_score(infected_mask)
    zone_score, zones_infected = compute_zone_score(infected_mask)

    # Step 4: weighted combination
    final_score = (W_AREA   * area_score   +
                   W_COLOR  * color_score  +
                   W_SPREAD * spread_score +
                   W_ZONE   * zone_score)

    infected_pct = area_score   # area score IS the infected percentage

    return {
        "severity_score":  round(final_score,   2),
        "severity_label":  severity_label(final_score),
        "infected_pct":    round(infected_pct,  2),
        "spot_count":      num_spots,
        "zones_infected":  zones_infected,
        "breakdown": {
            "area_score":   round(area_score,   2),
            "color_score":  round(color_score,  2),
            "spread_score": round(spread_score, 2),
            "zone_score":   round(zone_score,   2),
        },
    }


# ══════════════════════════════════════════════════════════════════════════════
# ❽  VISUALISATION  (for demo + report)
# ══════════════════════════════════════════════════════════════════════════════

def visualise_severity(img_bgr: np.ndarray,
                        result: dict,
                        disease: str = "",
                        save_path: str = None) -> np.ndarray:
    """
    Creates a side-by-side panel:
      Original | Leaf mask | Infected overlay | Score breakdown

    Returns the composite image (BGR numpy array).
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    h, w  = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Recompute masks for visualisation
    leaf_mask     = segment_leaf(img_bgr)
    infected_mask = detect_infected_region(img_bgr, leaf_mask)

    # Overlay: green = healthy leaf, red = infected
    overlay = img_rgb.copy()
    overlay[leaf_mask     > 0] = (overlay[leaf_mask     > 0] * 0.6 +
                                   np.array([0, 180, 0])     * 0.4).astype(np.uint8)
    overlay[infected_mask > 0] = (overlay[infected_mask > 0] * 0.4 +
                                   np.array([220, 50, 50])   * 0.6).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor("#1e1e1e")

    axes[0].imshow(img_rgb)
    axes[0].set_title("Original", color="white", fontsize=12)
    axes[0].axis("off")

    axes[1].imshow(overlay)
    green_patch = mpatches.Patch(color=(0, 180/255, 0), label="Healthy tissue")
    red_patch   = mpatches.Patch(color=(220/255, 50/255, 50/255), label="Infected region")
    axes[1].legend(handles=[green_patch, red_patch],
                   loc="lower left", fontsize=9,
                   facecolor="#333", labelcolor="white")
    axes[1].set_title("Segmentation overlay", color="white", fontsize=12)
    axes[1].axis("off")

    # Score breakdown bar chart
    bd      = result["breakdown"]
    labels  = ["Area\n(×0.5)", "Color\n(×0.3)", "Spread\n(×0.2)"]
    raw     = [bd["area_score"], bd["color_score"], bd["spread_score"]]
    contrib = [bd["area_score"]  * W_AREA,
               bd["color_score"] * W_COLOR,
               bd["spread_score"]* W_SPREAD]
    colors  = ["#E07B54", "#E0C454", "#54A0E0"]

    ax = axes[2]
    ax.set_facecolor("#2a2a2a")
    bars = ax.bar(labels, contrib, color=colors, edgecolor="white",
                  linewidth=0.5, width=0.5)
    for bar, raw_val in zip(bars, raw):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f"{raw_val:.1f}",
                ha="center", va="bottom", color="white", fontsize=10)

    ax.set_ylim(0, 55)
    ax.set_ylabel("Weighted contribution", color="white")
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#555")

    label  = result["severity_label"]
    score  = result["severity_score"]
    color_map = {"Mild": "#54C054", "Moderate": "#E0C454",
                 "Severe": "#E07B54", "Critical": "#CC3333"}
    ax.set_title(
        f"{disease}  |  {label}  ({score:.1f}%)\n"
        f"Infected: {result['infected_pct']:.1f}%  •  "
        f"Spots: {result['spot_count']}",
        color=color_map.get(label, "white"), fontsize=11, fontweight="bold"
    )

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight",
                    facecolor="#1e1e1e")
        print(f"  ✓ Visualisation saved → {save_path}")
    plt.show()
    plt.close()

    return overlay


# ══════════════════════════════════════════════════════════════════════════════
# ❾  STAGE 1 + STAGE 2 COMBINED PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def load_stage1_model():
    """Load EfficientNet-B0 classifier from Stage 1."""
    import torch
    import torch.nn as nn
    from torchvision import models, transforms
    from PIL import Image as PILImage

    MODEL_PATH = "models/classifier.pth"
    CLASS_MAP  = "models/class_map.json"

    if not Path(MODEL_PATH).exists():
        print("  [WARN] Stage 1 model not found — skipping disease prediction")
        return None, None, None

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    with open(CLASS_MAP) as f:
        cmap = json.load(f)
    idx_to_class = {int(k): v for k, v in cmap["idx_to_class"].items()}
    num_classes  = len(idx_to_class)

    model = models.efficientnet_b0(weights=None)
    in_f  = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, num_classes))
    ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval().to(DEVICE)

    tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                              [0.229, 0.224, 0.225]),
    ])
    return model, idx_to_class, tf


def predict_full(image_path: str, save_viz: bool = False) -> dict:
    """
    Full pipeline: Stage 1 (disease) + Stage 2 (severity).
    """
    import torch
    from PIL import Image as PILImage

    # Load image
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read: {image_path}")

    result = {"image": image_path}

    # ── Stage 1: disease classification ──────────────────────────────────────
    s1_model, idx_to_class, tf = load_stage1_model()
    if s1_model is not None:
        DEVICE = next(s1_model.parameters()).device
        pil_img = PILImage.open(image_path).convert("RGB")
        inp     = tf(pil_img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs      = torch.softmax(s1_model(inp), dim=1)[0]
            conf, idx  = probs.max(0)
        result["disease"]    = idx_to_class[idx.item()]
        result["confidence"] = f"{conf.item()*100:.1f}%"
    else:
        result["disease"]    = "Unknown"
        result["confidence"] = "N/A"

    # ── Stage 2: severity ─────────────────────────────────────────────────────
    # Skip severity if leaf is healthy
    if result["disease"].lower() == "healthy":
        result.update({
            "severity_score": 0.0,
            "severity_label": "Healthy",
            "infected_pct":   0.0,
            "spot_count":     0,
            "breakdown": {"area_score": 0.0,
                          "color_score": 0.0,
                          "spread_score": 0.0},
        })
    else:
        severity = compute_severity(img_bgr)
        result.update(severity)

    # ── Visualise ─────────────────────────────────────────────────────────────
    if save_viz:
        output_dir = Path("stage2_severity_output")
        output_dir.mkdir(exist_ok=True)
        viz_path = output_dir / (Path(image_path).stem + "_severity.png")
        visualise_severity(img_bgr, result,
                           disease=result["disease"],
                           save_path=str(viz_path))

    return result


# ══════════════════════════════════════════════════════════════════════════════
# ❿  BATCH EVALUATION on stage2_dataset/test/
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_on_test_set(test_dir: str = "stage2_dataset/test",
                          output_json: str = "models/stage2_eval_results.json"):
    """
    Runs severity on all test images, saves results + severity distribution.
    """
    from collections import Counter

    test_path = Path(test_dir) / "images"
    if not test_path.exists():
        print(f"[ERROR] Test dir not found: {test_path}")
        return

    img_files = sorted(test_path.glob("*.jpg"))
    print(f"Evaluating severity on {len(img_files)} test images...")

    results  = []
    sev_dist = Counter()

    for i, img_path in enumerate(img_files):
        img_bgr  = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        sev = compute_severity(img_bgr)
        sev["image"] = img_path.name
        results.append(sev)
        sev_dist[sev["severity_label"]] += 1

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(img_files)}] done...")

    os.makedirs("models", exist_ok=True)
    with open(output_json, "w") as f:
        json.dump(results, f, indent=2)

    # Summary
    print(f"\n{'═'*55}")
    print(f"  STAGE 2 EVALUATION — {len(results)} test images")
    print(f"{'═'*55}")
    avg_score = np.mean([r["severity_score"] for r in results])
    avg_area  = np.mean([r["infected_pct"]   for r in results])
    avg_spots = np.mean([r["spot_count"]     for r in results])
    print(f"  Avg severity score : {avg_score:.1f}")
    print(f"  Avg infected area  : {avg_area:.1f}%")
    print(f"  Avg spot count     : {avg_spots:.1f}")
    print(f"\n  Severity distribution:")
    for sev in ["Mild", "Moderate", "Severe", "Critical"]:
        n   = sev_dist.get(sev, 0)
        pct = n / len(results) * 100 if results else 0
        bar = "▓" * int(pct / 3)
        print(f"    {sev:<10}: {n:>4}  ({pct:.1f}%)  {bar}")
    print(f"\n  ✓ Results saved → {output_json}")
    print(f"{'═'*55}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # Modified for Jupyter notebook: directly process stage2_dataset/train
    batch_folder = "stage2_dataset/train/images"
    save_viz = True  # Set to False if you don't want visualizations

    output_dir = Path("severity_output_pt2")
    output_dir.mkdir(exist_ok=True)

    # OPTIONAL: Move any old severity images into folder
    moved_count = 0
    for f in Path(".").glob("*_severity.png"):
        shutil.move(str(f), str(output_dir / f.name))
        moved_count += 1
    if moved_count > 0:
        print(f"Moved {moved_count} old severity images to stage2_severity_output/")

    results = []
    for img_path in sorted(Path(batch_folder).glob("*.jpg")):
        r = predict_full(str(img_path), save_viz=save_viz)

        print(f"  {img_path.name:30s} → "
              f"{r.get('disease','?'):15s} | "
              f"{r['severity_label']:10s} "
              f"({r['severity_score']:.1f}%)")

        results.append(r)

    # Save JSON also inside same folder (cleaner)
    with open(output_dir / "batch_results.json", "w") as f:
        json.dump(results, f, indent=2)

    print(f"\n✓ All outputs saved in: stage2_severity_output/")

  ✓ Visualisation saved → stage2_severity_output/train_00000_severity.png
  train_00000.jpg                → DeficiencyLeaf  | Moderate   (26.1%)


/var/folders/fq/fn55_cv53qqfd0n46tbgzj800000gn/T/ipykernel_19808/3999942184.py:436: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  ✓ Visualisation saved → stage2_severity_output/train_00001_severity.png
  train_00001.jpg                → Greening        | Severe     (62.7%)
  ✓ Visualisation saved → stage2_severity_output/train_00002_severity.png
  train_00002.jpg                → Anthracnose     | Mild       (22.0%)
  ✓ Visualisation saved → stage2_severity_output/train_00003_severity.png
  train_00003.jpg                → DeficiencyLeaf  | Severe     (63.6%)
  ✓ Visualisation saved → stage2_severity_output/train_00004_severity.png
  train_00004.jpg                → Anthracnose     | Mild       (24.1%)
  ✓ Visualisation saved → stage2_severity_output/train_00005_severity.png
  train_00005.jpg                → Anthracnose     | Mild       (21.0%)
  ✓ Visualisation saved → stage2_severity_output/train_00006_severity.png
  train_00006.jpg                → Greening        | Mild       (16.2%)
  ✓ Visualisation saved → stage2_severity_output/train_00007_severity.png
  train_00007.jpg                → Canker         

In [1]:
import os
import json
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models, transforms
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================

MODEL_PATH = "models_v2/best_model.pth"
THRESH_PATH = "models_v2/thresholds.json"
CLASS_MAP_PATH = "models_v2/class_map.json"

DATASET_ROOT = "dataset_balanced"
OUTPUT_ROOT = "severity_outputs"

IMG_SIZE = 300
DEFAULT_THRESH = 0.5

ALL_CLASSES = [
    "Anthracnose", "BacterialBlight", "BlackSpot", "Canker",
    "CurlVirus", "DeficiencyLeaf", "DryLeaf", "Greening",
    "Healthy", "LeafMiner", "Melanose", "SootyMould", "SpiderMites"
]

SEVERITY_THRESHOLDS = {
    "Mild": (0, 25),
    "Moderate": (25, 50),
    "Severe": (50, 75),
    "Critical": (75, 100),
}

W_AREA = 0.40
W_COLOR = 0.25
W_SPREAD = 0.20
W_ZONE = 0.15

HEALTHY_LAB = np.array([128, 110, 145], dtype=np.float32)
HSV_LOWER_GREEN = np.array([25, 30, 30])
HSV_UPPER_GREEN = np.array([95, 255, 255])

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

# =========================================================
# MODEL LOADING
# =========================================================

def load_model():
    ckpt = torch.load(MODEL_PATH, map_location=DEVICE)

    model = models.efficientnet_b3(weights=None)
    in_feats = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_feats, 512),
        nn.SiLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, len(ALL_CLASSES)),
    )
    model.load_state_dict(ckpt["model_state"])
    model.eval().to(DEVICE)

    thresholds = np.full(len(ALL_CLASSES), DEFAULT_THRESH)
    if Path(THRESH_PATH).exists():
        with open(THRESH_PATH) as f:
            tdict = json.load(f)
        thresholds = np.array([tdict.get(c, DEFAULT_THRESH) for c in ALL_CLASSES])

    tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.6753, 0.7039, 0.5966],
            std=[0.3517, 0.3215, 0.4195],
        ),
    ])

    return model, thresholds, tf

# =========================================================
# STAGE 1 MULTI-LABEL PREDICTION
# =========================================================

@torch.no_grad()
def predict_diseases(model, tf, image_path, thresholds):
    pil_img = Image.open(image_path).convert("RGB")
    x = tf(pil_img).unsqueeze(0).to(DEVICE)

    logits = model(x)
    probs = torch.sigmoid(logits).cpu().numpy()[0]

    detected = []
    for cls, prob, thr in zip(ALL_CLASSES, probs, thresholds):
        if prob >= thr:
            detected.append({
                "disease": cls,
                "confidence": float(prob)
            })

    detected.sort(key=lambda d: -d["confidence"])

    non_healthy = [d for d in detected if d["disease"] != "Healthy"]
    if non_healthy:
        detected = non_healthy

    return detected, probs

# =========================================================
# LEAF + INFECTED REGION MASKS
# =========================================================

def segment_leaf(img_bgr):
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(img_hsv, HSV_LOWER_GREEN, HSV_UPPER_GREEN)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)

    leaf_ratio = mask.sum() / (mask.size * 255)
    if leaf_ratio < 0.10:
        mask = np.ones(img_bgr.shape[:2], dtype=np.uint8) * 255

    return mask

def detect_infected_region(img_bgr, leaf_mask):
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    healthy_mask = cv2.inRange(img_hsv, HSV_LOWER_GREEN, HSV_UPPER_GREEN)

    infected = cv2.bitwise_and(cv2.bitwise_not(healthy_mask), leaf_mask)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    infected = cv2.morphologyEx(infected, cv2.MORPH_CLOSE, kernel, iterations=2)
    infected = cv2.morphologyEx(infected, cv2.MORPH_OPEN, kernel, iterations=1)

    return infected

# =========================================================
# GRAD-CAM
# =========================================================

class GradCAM:
    def __init__(self, model):
        self.model = model
        self.activations = None
        self.gradients = None

        target_layer = self.model.features[-1]
        target_layer.register_forward_hook(self._forward_hook)
        target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inp, out):
        self.activations = out

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, x, class_idx):
        self.model.zero_grad()
        logits = self.model(x)
        score = logits[:, class_idx].sum()
        score.backward(retain_graph=True)

        grads = self.gradients[0]
        acts = self.activations[0]

        weights = grads.mean(dim=(1, 2), keepdim=True)
        cam = (weights * acts).sum(dim=0)
        cam = torch.relu(cam)

        cam = cam.detach().cpu().numpy()
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))

        if cam.max() > 0:
            cam = cam / cam.max()

        return cam

# =========================================================
# SEVERITY SIGNALS
# =========================================================

def compute_area_score(infected_mask, leaf_mask):
    leaf_pixels = max(1, (leaf_mask > 0).sum())
    infected_pixels = (infected_mask > 0).sum()
    return min(100.0, float(infected_pixels / leaf_pixels * 100))

def compute_color_score(img_bgr, disease_mask):
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    pixels = img_lab[disease_mask > 0]
    if len(pixels) == 0:
        return 0.0

    delta_e = np.linalg.norm(pixels - HEALTHY_LAB, axis=1)
    mean_delta = delta_e.mean()
    return min(100.0, float(mean_delta / 260.0 * 100))

def compute_spread_score(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    min_area = mask.size * 0.0001
    contours = [c for c in contours if cv2.contourArea(c) > min_area]
    num_spots = len(contours)

    if num_spots == 0:
        return 0.0, 0

    centroids = []
    for c in contours:
        M = cv2.moments(c)
        if M["m00"] > 0:
            cx = M["m10"] / M["m00"]
            cy = M["m01"] / M["m00"]
            centroids.append((cx, cy))

    if len(centroids) < 2:
        avg_dist = 0.0
    else:
        distances = []
        for i in range(len(centroids)):
            for j in range(i + 1, len(centroids)):
                dx = centroids[i][0] - centroids[j][0]
                dy = centroids[i][1] - centroids[j][1]
                distances.append(np.sqrt(dx**2 + dy**2))
        avg_dist = np.mean(distances)

    h, w = mask.shape
    max_dist = np.sqrt(h**2 + w**2) / 2
    count_score = min(1.0, num_spots / 20.0)
    dist_score = min(1.0, avg_dist / max_dist) if max_dist > 0 else 0.0

    spread_score = (0.5 * count_score + 0.5 * dist_score) * 100
    return float(spread_score), num_spots

def compute_zone_score(mask, grid=3):
    h, w = mask.shape
    total_zones = grid * grid
    zone_h = h // grid
    zone_w = w // grid
    zones_infected = 0

    for row in range(grid):
        for col in range(grid):
            y1 = row * zone_h
            y2 = y1 + zone_h if row < grid - 1 else h
            x1 = col * zone_w
            x2 = x1 + zone_w if col < grid - 1 else w
            zone = mask[y1:y2, x1:x2]

            if zone.size > 0 and (zone > 0).mean() > 0.01:
                zones_infected += 1

    zone_score = (zones_infected / total_zones) * 100
    return float(zone_score), zones_infected

def severity_label(score):
    for label, (lo, hi) in SEVERITY_THRESHOLDS.items():
        if lo <= score < hi:
            return label
    return "Critical"

# =========================================================
# MULTI-DISEASE SEVERITY
# =========================================================

def compute_disease_specific_mask(cam, infected_mask, cam_thresh=0.35):
    cam_mask = (cam >= cam_thresh).astype(np.uint8) * 255
    disease_mask = cv2.bitwise_and(cam_mask, infected_mask)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    disease_mask = cv2.morphologyEx(disease_mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    disease_mask = cv2.morphologyEx(disease_mask, cv2.MORPH_OPEN, kernel, iterations=1)

    return disease_mask

def compute_one_disease_severity(img_bgr, leaf_mask, disease_mask):
    area_score = compute_area_score(disease_mask, leaf_mask)
    color_score = compute_color_score(img_bgr, disease_mask)
    spread_score, spot_count = compute_spread_score(disease_mask)
    zone_score, zones_infected = compute_zone_score(disease_mask)

    final_score = (
        W_AREA * area_score +
        W_COLOR * color_score +
        W_SPREAD * spread_score +
        W_ZONE * zone_score
    )

    return {
        "severity_score": round(final_score, 2),
        "severity_label": severity_label(final_score),
        "infected_pct": round(area_score, 2),
        "spot_count": spot_count,
        "zones_infected": zones_infected,
        "breakdown": {
            "area_score": round(area_score, 2),
            "color_score": round(color_score, 2),
            "spread_score": round(spread_score, 2),
            "zone_score": round(zone_score, 2),
        }
    }

def compute_multidisease_severity(image_path, model, thresholds, tf, gradcam):
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")

    img_bgr = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))
    leaf_mask = segment_leaf(img_bgr)
    infected_mask = detect_infected_region(img_bgr, leaf_mask)

    detected, probs = predict_diseases(model, tf, str(image_path), thresholds)
    x = tf(Image.open(image_path).convert("RGB")).unsqueeze(0).to(DEVICE)

    per_disease = []
    union_mask = np.zeros_like(infected_mask)

    for item in detected:
        disease = item["disease"]
        conf = item["confidence"]
        class_idx = ALL_CLASSES.index(disease)

        cam = gradcam.generate(x, class_idx)
        disease_mask = compute_disease_specific_mask(cam, infected_mask, cam_thresh=0.35)

        if (disease_mask > 0).sum() == 0:
            continue

        sev = compute_one_disease_severity(img_bgr, leaf_mask, disease_mask)
        sev["disease"] = disease
        sev["confidence"] = round(conf, 4)
        per_disease.append(sev)

        union_mask = cv2.bitwise_or(union_mask, disease_mask)

    overall = compute_one_disease_severity(
        img_bgr,
        leaf_mask,
        union_mask if union_mask.sum() > 0 else infected_mask
    )

    return {
        "image": str(image_path),
        "detected_diseases": detected,
        "overall_severity": overall,
        "per_disease_severity": per_disease,
        "all_probs": {cls: round(float(p), 4) for cls, p in zip(ALL_CLASSES, probs)}
    }

# =========================================================
# VISUALIZE
# =========================================================

def visualize_multidisease_result(image_path, result, save_path=None):
    img_bgr = cv2.imread(str(image_path))
    img_bgr = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(img_rgb)
    axes[0].set_title("Input Leaf")
    axes[0].axis("off")

    text_lines = []
    text_lines.append("Detected diseases:")
    for d in result["per_disease_severity"]:
        text_lines.append(
            f'{d["disease"]}: {d["severity_label"]} ({d["severity_score"]}%), conf={d["confidence"]:.2f}'
        )

    text_lines.append("")
    text_lines.append(
        f'Overall: {result["overall_severity"]["severity_label"]} '
        f'({result["overall_severity"]["severity_score"]}%)'
    )

    axes[1].axis("off")
    axes[1].text(0.01, 0.98, "\n".join(text_lines), va="top", fontsize=12)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

# =========================================================
# BATCH PROCESS DATASET
# =========================================================

def process_dataset():
    model, thresholds, tf = load_model()
    gradcam = GradCAM(model)

    output_root = Path(OUTPUT_ROOT)
    output_root.mkdir(exist_ok=True)

    image_exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

    splits = ["train", "val", "test"]

    for split in splits:
        split_dir = Path(DATASET_ROOT) / split
        if not split_dir.exists():
            print(f"Skipping missing split: {split_dir}")
            continue

        split_output = output_root / split
        split_output.mkdir(parents=True, exist_ok=True)

        image_paths = []
        for path in split_dir.rglob("*"):
            if path.suffix in image_exts:
                image_paths.append(path)

        print(f"\nProcessing {split} → {len(image_paths)} images")

        for i, image_path in enumerate(sorted(image_paths), 1):
            try:
                result = compute_multidisease_severity(
                    image_path, model, thresholds, tf, gradcam
                )

                rel_name = image_path.stem
                json_path = split_output / f"{rel_name}_severity.json"
                viz_path = split_output / f"{rel_name}_viz.png"


                visualize_multidisease_result(image_path, result, save_path=viz_path)

                print(
                    f"[{split} {i}/{len(image_paths)}] "
                    f"{image_path.name} → "
                    f'{result["overall_severity"]["severity_label"]} '
                    f'({result["overall_severity"]["severity_score"]}%)'
                )

            except Exception as e:
                print(f"Error processing {image_path}: {e}")

# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":
    process_dataset()


Processing train → 8340 images
[train 1/8340] Anthracnose_train_00003_c00459.jpg → Mild (17.88%)
[train 2/8340] Anthracnose_train_00005_c00585.jpg → Mild (12.34%)
[train 3/8340] Anthracnose_train_00008_c00472.jpg → Moderate (29.04%)
[train 4/8340] Anthracnose_train_00010_c00235.jpg → Mild (12.39%)
[train 5/8340] Anthracnose_train_00011_c00725.jpg → Mild (20.16%)
[train 6/8340] Anthracnose_train_00012_c00298.jpg → Mild (10.1%)
[train 7/8340] Anthracnose_train_00017_c00761.jpg → Moderate (28.01%)
[train 8/8340] Anthracnose_train_00019_c00525.jpg → Mild (9.03%)
[train 9/8340] Anthracnose_train_00020_c00654.jpg → Mild (16.42%)
[train 10/8340] Anthracnose_train_00021_c00094.jpg → Mild (22.89%)
[train 11/8340] Anthracnose_train_00023_c00082.jpg → Mild (13.1%)
[train 12/8340] Anthracnose_train_00024_c00333.jpg → Mild (19.68%)
[train 13/8340] Anthracnose_train_00025_c00605.jpg → Mild (11.38%)
[train 14/8340] Anthracnose_train_00026_c00286.jpg → Mild (24.24%)
[train 15/8340] Anthracnose_train_